### Building a chatBot
in notebook we will go over an example of how to design and implement an LLM powered chatbot this chatbot will able to have a conversation and remain and remember previous interactions.
note that the chatBot that we will only use the language model to have a conversation there are several other related concepts that may be looking after some time
*    **Conversational RAG:** enable a chatbot experience over an external data
*    **Agents:** built a chatBot that can take actions
 so in this notebook we will cover the basics that will helpful for those two more advanced concepts

In [1]:
import os
from http.client import responses

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sympy.parsing.sympy_parser import null

# from LanguageTranslation.serve import prompt

#load the groq api key from the .env file
load_dotenv()
groq_api_key = os.getenv("GROG_API_KEY")

groq_api_key

C:\Users\ybalasaraswa\OneDrive - OpenText\Desktop\deep learning\LangChain\.myvenv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
C:\Users\ybalasaraswa\OneDrive - OpenText\Desktop\deep learning\LangChain\.myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'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'

In [2]:
#load the groq chat model
model = ChatGroq(api_key=groq_api_key,
                 model="openai/gpt-oss-120b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000012B771D42F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000012B771D4D70>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage, AIMessage

model.invoke(
    [
        HumanMessage(content="hay, hi my name is Yugi")
        # AIMessage(),
        # HumanMessage(),
    ]
)

AIMessage(content='Hey there, Yugi! 👋 Nice to meet you. How’s your day going? Anything fun or interesting you’re up to right now?', additional_kwargs={'reasoning_content': 'The user says: "hay, hi my name is Yugi". Likely they are greeting and introducing themselves. Should respond friendly, ask about them, maybe about Yugi (maybe reference to Yu-Gi-Oh). We can respond with a greeting, ask how they are, maybe ask about interests. Keep it friendly.'}, response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 79, 'total_tokens': 185, 'completion_time': 0.2405767, 'completion_tokens_details': {'reasoning_tokens': 67}, 'prompt_time': 0.015845129, 'prompt_tokens_details': None, 'queue_time': 0.707851791, 'total_time': 0.256421829}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4727af4560', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c23b5-6827-7151-976d-b8dd9f9a5d8c-0', tool_calls=[], 

In [4]:
#now lets see weather the model can remember the previous messages
model.invoke(
    [
        HumanMessage(content="hay, hi my name is Yugi"),
        AIMessage(content="Hey there, Yugi! 👋 Nice to meet you. How can I help you today?"),
        HumanMessage(content="what is my name?"),
    ]
)

AIMessage(content='You introduced yourself as **Yugi**. 😊', additional_kwargs={'reasoning_content': 'The user asks "what is my name?" Previously they said "hay, hi my name is Yugi". So answer: Yugi. Probably confirm.'}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 114, 'total_tokens': 165, 'completion_time': 0.108670471, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.004643767, 'prompt_tokens_details': None, 'queue_time': 0.054124892, 'total_time': 0.113314238}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c23b5-6c89-70d2-8072-85645bc5d5a2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 114, 'output_tokens': 51, 'total_tokens': 165, 'output_token_details': {'reasoning': 32}})

### Message History
to enable the chatBot to remember the previous interactions we need to maintain a message history this history will store all the messages exchanged between the user and the chatBot

In [5]:
!pip install langchain_community

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def getSessionHistory(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(runnable=model,
                                                  get_session_history=getSessionHistory)


In [7]:
config={"configurable": {"session_id": "user_123"}}

In [8]:
with_message_history.invoke(
    [HumanMessage(content="what is my name?")],
    config=config
)

AIMessage(content='I don’t have any information about your name. If you’d like me to address you a certain way, just let me know!', additional_kwargs={'reasoning_content': 'The user asks "what is my name?" There\'s no prior context. The system says "You are ChatGPT..." There\'s no instruction to preserve privacy. According to policy, we cannot guess personal info. We should respond that we don\'t know the name, ask if they\'d like to tell.'}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 76, 'total_tokens': 170, 'completion_time': 0.203874274, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.014998051, 'prompt_tokens_details': None, 'queue_time': 0.055421578, 'total_time': 0.218872325}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_626f3fc5e0', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c23b5-733c-71c1-9317-ef3c213990fa-0', tool_calls=[], invalid

In [9]:
#change the Config with different session id
config1={"configurable": {"session_id": "user_456"}}

with_message_history.invoke(
    [
        HumanMessage(content="hay, what is my name?")
    ],
    config=config1
)


AIMessage(content='I don’t actually know your name—could you let me know what you’d like me to call you?', additional_kwargs={'reasoning_content': 'The user asks "hay, what is my name?" There\'s no prior conversation. The user asks for personal info. The assistant doesn\'t know the name. According to policy, we must respond that we don\'t know. We can ask politely for clarification. Also note the greeting "hay" maybe "hey". We can respond: "I don\'t know your name, could you tell me?" Should comply.'}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 78, 'total_tokens': 189, 'completion_time': 0.234129069, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.003215269, 'prompt_tokens_details': None, 'queue_time': 0.053729088, 'total_time': 0.237344338}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--0

In [10]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant answer everything what user is asking."),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain = prompt | model | parser

In [12]:
chain.invoke({"messages": [HumanMessage(content="who is the CM for telangana?")]})

'The current Chief Minister of\u202fTelangana is **K.\u202fChandrashekar\u202fRao** (often abbreviated as\u202fKCR). He heads the Bharat\u202fRashtra\u202fSamithi (BRS)—the party formerly known as the Telangana\u202fRashtra\u202fSamithi (TRS)—and has been in office since the state’s formation in 2014, winning re‑election in the 2018 and 2024 state elections.'

In [13]:
with_message_history = RunnableWithMessageHistory(chain,
                                                  getSessionHistory)
config={"configurable": {"session_id": "user_123"}}
response = with_message_history.invoke(
    [HumanMessage(content="who is the CM for telangana?")],
    config=config
)
response

'The Chief Minister of\u202fTelangana is **K.\u202fChandrashekar\u202fRao** (often referred to as\u202fKCR). He leads the state’s government as the head of the Bharat\u202fRashtra\u202fSamithi (BRS) party.'